# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [45]:
!git clone https://github.com/Nikita-Sudarshan/flyrank-ml-starter.git
%cd /content/flyrank-ml-starter

Cloning into 'flyrank-ml-starter'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 138 (delta 47), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.89 MiB | 10.51 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/flyrank-ml-starter


In [46]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


In [47]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [48]:
print(df.dtypes)

content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            int64
age_tier      

In [49]:
for col in df.columns:
    if "label" in col.lower() or "target" in col.lower() or "declin" in col.lower():
        print(col, df[col].value_counts(dropna=False).to_dict())

In [50]:
print(df["trend_direction"].value_counts(dropna=False))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [51]:
print(df[["trend_direction", "trend_pct"]].head(20))

   trend_direction  trend_pct
0             down      -41.4
1             down      -57.7
2             down      -60.9
3           stable      -13.8
4             down      -34.7
5             down      -38.9
6             down      -92.3
7           stable        0.6
8             down      -58.8
9             down      -29.2
10          stable       19.0
11             new        NaN
12              up      277.8
13          stable       10.4
14            down      -64.9
15              up      675.0
16            down      -39.1
17            down      -20.2
18            down      -56.9
19            down      -67.7


In [52]:
!grep -R "trend_direction" -n docs skills work scripts --exclude="*.ipynb" | head -50

docs/data-dictionary.md:11:2. **The label comes from `trend_direction`.** The pipeline defines
docs/data-dictionary.md:12:   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
docs/data-dictionary.md:99:| `trend_direction` | `new`, `flat`, `up`, `down`, `stable` | last-30d vs prev-30d impressions: `new` = prev 0 & last > 0; `flat` = both 0; `up` > +20%; `down` < −20%; else `stable`. **Label source — never a feature** |
docs/data-dictionary.md:107:| `is_declining_label` | **The target.** 1 when `trend_direction == "down"` (16,262 rows = 54.2%), else 0 |
docs/ml-intern-dataset-and-lane-guide.md:188:Those product decisions are deliberately **not** in your data. Your data ships observable search and engagement signals, plus transparent derived buckets (tiers, `trend_direction`, `trend_pct`, ctr, rates) — and nothing else. This is on purpose, so you discover signal from evidence instead of accidentally copying the product's own answers.
docs/ml-intern-d

In [53]:
!grep -R "trend_pct" -n docs skills work scripts --exclude="*.ipynb" | head -50

docs/data-dictionary.md:10:   `ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `trend_pct`.
docs/data-dictionary.md:12:   `is_declining_label = (trend_direction == "down")`, so `trend_direction` and `trend_pct`
docs/data-dictionary.md:86:| `trend_pct` | `(impressions_last_30d − impressions_prev_30d) / impressions_prev_30d × 100` | 1 decimal. Blank when `impressions_prev_30d = 0` (3,388 rows; the prep step fills blanks with 0). **Label source — never a feature** |
docs/ml-intern-dataset-and-lane-guide.md:179:| Derived measurements | Numbers calculated from observed signals | `trend_pct`, age/freshness tiers, position tier | Usually fine, if calculated only from the feature window |
docs/ml-intern-dataset-and-lane-guide.md:188:Those product decisions are deliberately **not** in your data. Your data ships observable search and engagement signals, plus transparent derived buckets (tiers, `trend_direction`, `trend_pct`, ctr, rates) — and nothing else. This is on purpose, so you di

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [54]:
# ### Method choice

# I will use Logistic Regression to predict whether a content item is declining.

# The target is `is_declining_label`, defined as `trend_direction == "down"`. The target rate is 54.2% in the full dataset, so I will report the base rate alongside the model results.

# Logistic Regression is a suitable first model for this yes/no observed-label question because it is simple, reproducible, and interpretable. Its predicted probabilities can also be used to rank content for review, which matches the action-prioritization goal from Week 4.

# I will not use `trend_direction` or `trend_pct` as features because they directly define or derive the target and would create label leakage. I will also exclude the last-30-day and previous-30-day impression fields because the target is defined from their change.

# The first model will be evaluated against the Week-4 baseline using precision@20 on the same grouped test split.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [55]:
# ### Split design

# I will use a grouped train/test split by `client_id`, with 80% of clients in the training set and 20% in the test set.

# Grouping by client prevents content from the same client appearing in both training and test data. This is more honest for the capstone question because the test set represents clients that were not used to fit the model.

# I will use a fixed random seed so the split is reproducible.

# The test set will be held out until model evaluation. The same test data and evaluation metric will be used when comparing the Decision Tree with the Week-4 baseline.

In [56]:
from sklearn.model_selection import GroupShuffleSplit

X = df.drop(columns=["trend_direction"])
y = df["trend_direction"].eq("down").astype(int)

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", X_train["client_id"].nunique())
print("Test clients:", X_test["client_id"].nunique())

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7


In [57]:
train_clients = set(X_train["client_id"])
test_clients = set(X_test["client_id"])

overlap = train_clients.intersection(test_clients)

print("Client overlap:", len(overlap))

Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [58]:
excluded_columns = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_id",
    "client_id",
    "is_declining_label"
]

feature_columns = [
    col for col in X_train.columns
    if col not in excluded_columns
]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 38
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'clicks_last_30d', 'sessions_last_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


In [59]:
for col in excluded_columns:
    if col in feature_columns:
        print("WARNING:", col, "is still included")

In [60]:
categorical_features = X_train[feature_columns].select_dtypes(
    include=["object"]
).columns.tolist()

numeric_features = X_train[feature_columns].select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)

Categorical features:
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'clicks_last_30d', 'sessions_last_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


In [61]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            categorical_features
        )
    ]
)

X_train_features = X_train[feature_columns]
X_test_features = X_test[feature_columns]

X_train_processed = preprocessor.fit_transform(X_train_features)
X_test_processed = preprocessor.transform(X_test_features)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (23837, 69)
Processed test shape: (6163, 69)


In [62]:
!cat skills/training-honest-models/SKILL.md

---
name: training-honest-models
description: Trains a first model the honest way — method chosen to fit the question, compared against the baseline on the same split and metric, errors read before scores are believed. Use when moving from a rule baseline to a learned model, or when reviewing a model that reports only a single score.
---

# Training honest models

The model is the easy part. The honesty is the work: same data, same split, same metric as the
baseline — then read the errors before believing the score.

## Choose the method to fit the question

| Question shape | Start with | Because |
|---|---|---|
| yes/no with an observed label | Logistic Regression, then Random Forest | readable → stronger |
| "which first?" ranking | any classifier's probability, evaluated at precision@K | ranking needs scores, not labels |
| grouping items | K-Means (pick k with silhouette), then NAME clusters after inspecting them | unsupervised needs human naming |
| "what drives X?" | simple mode

In [63]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_processed, y_train)

print("Model trained successfully.")

Model trained successfully.


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [64]:
y_test_proba = model.predict_proba(X_test_processed)[:, 1]
y_test_pred = model.predict(X_test_processed)

print("Predictions generated:", len(y_test_pred))

Predictions generated: 6163


In [65]:
from sklearn.metrics import precision_score

# Rank test items by predicted probability of declining
model_ranked = X_test[["content_id"]].copy()
model_ranked["actual"] = y_test.values
model_ranked["predicted_probability"] = y_test_proba

model_top20 = model_ranked.sort_values(
    "predicted_probability",
    ascending=False
).head(20)

model_precision_at_20 = model_top20["actual"].mean()

print("Model Precision@20:", model_precision_at_20)
print("Declining items in top 20:", model_top20["actual"].sum())

Model Precision@20: 0.9
Declining items in top 20: 18


In [66]:
# Apply the Week-4 baseline rule to the held-out test set

baseline_test = X_test[[
    "content_id",
    "days_since_last_update",
    "impressions_90d"
]].copy()

baseline_test["actual"] = y_test.values

# Week-4 baseline score
baseline_test["score"] = (
    (baseline_test["days_since_last_update"] >= 181)
    & (baseline_test["impressions_90d"] >= 500)
).astype(int) * baseline_test["impressions_90d"]

# Rank using the same logic as Week 4
baseline_ranked = baseline_test.sort_values(
    by=["score", "impressions_90d"],
    ascending=[False, False]
)

baseline_top20 = baseline_ranked.head(20)

baseline_precision_at_20 = baseline_top20["actual"].mean()

print("Baseline Precision@20:", baseline_precision_at_20)
print(
    "Declining items in baseline top 20:",
    baseline_top20["actual"].sum()
)

Baseline Precision@20: 0.35
Declining items in baseline top 20: 7


In [67]:
base_rate = y_test.mean()

print("Test-set declining base rate:", base_rate)
print("Declining items in test set:", y_test.sum())
print("Total test items:", len(y_test))

Test-set declining base rate: 0.5109524582184002
Declining items in test set: 3149
Total test items: 6163


In [68]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "precision_at_20": [
        baseline_precision_at_20,
        model_precision_at_20
    ],
    "declining_in_top_20": [
        int(baseline_top20["actual"].sum()),
        int(model_top20["actual"].sum())
    ],
    "base_rate": [
        base_rate,
        base_rate
    ]
})

comparison

,method,precision_at_20,declining_in_top_20,base_rate
0,Week-4 baseline,0.35,7,0.510952
1,Logistic Regression,0.90,18,0.510952


In [69]:
# ### Model vs baseline

# On the grouped held-out test set, the Week-4 baseline achieved 35% precision@20, with 7 of its top 20 items actually declining. Logistic Regression achieved 90% precision@20, with 18 of its top 20 items actually declining.

# The test-set declining base rate was 51.1%. The Logistic Regression ranking was therefore substantially above the base rate and the Week-4 baseline on this split. This is an observed result on the held-out test clients, not evidence that the model will perform identically on future data.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [70]:
from sklearn.preprocessing import StandardScaler

In [71]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            categorical_features
        )
    ]
)

In [72]:
X_train_features = X_train[feature_columns]
X_test_features = X_test[feature_columns]

X_train_processed = preprocessor.fit_transform(X_train_features)
X_test_processed = preprocessor.transform(X_test_features)

In [73]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_test_proba = model.predict_proba(X_test_processed)[:, 1]
y_test_pred = model.predict(X_test_processed)

In [74]:
model_ranked = X_test[["content_id"]].copy()
model_ranked["actual"] = y_test.values
model_ranked["predicted_probability"] = y_test_proba

model_top20 = model_ranked.sort_values(
    "predicted_probability",
    ascending=False
).head(20)

model_precision_at_20 = model_top20["actual"].mean()

print("Model Precision@20:", model_precision_at_20)
print("Declining items in top 20:", model_top20["actual"].sum())

Model Precision@20: 0.8
Declining items in top 20: 16


In [75]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "precision_at_20": [
        baseline_precision_at_20,
        model_precision_at_20
    ],
    "declining_in_top_20": [
        int(baseline_top20["actual"].sum()),
        int(model_top20["actual"].sum())
    ],
    "base_rate": [
        base_rate,
        base_rate
    ]
})

comparison

,method,precision_at_20,declining_in_top_20,base_rate
0,Week-4 baseline,0.35,7,0.510952
1,Logistic Regression,0.80,16,0.510952


In [76]:
# ### Model vs baseline

# On the grouped held-out test set, the Week-4 baseline achieved 35% precision@20, with 7 of its top 20 items actually declining. Logistic Regression achieved 80% precision@20, with 16 of its top 20 items actually declining.

# The test-set declining base rate was 51.1%. The Logistic Regression ranking was above both the base rate and the Week-4 baseline on this split. The result is an observed improvement on held-out clients, not a guarantee of future performance.

In [77]:
feature_names = preprocessor.get_feature_names_out()

coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": model.coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

top_features = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
).head(15)

top_features[["feature", "coefficient"]]

,feature,coefficient
15,numeric__clicks_last_30d,-2.722210
9,numeric__users_90d,-1.749021
8,numeric__sessions_90d,1.661381
6,numeric__clicks_90d,1.309335
68,categorical__position_tier_top_3,-1.065065
17,numeric__clicks_prev_30d,1.041474
13,numeric__days_with_impressions,0.739834
35,categorical__main_intent_navigational,-0.600359
42,categorical__model_used_gpt-5-mini,0.585606
31,categorical__content_type_feedly article,-0.538550


In [78]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test_processed,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=42
)

permutation_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

permutation_df.head(15)

,feature,importance_mean,importance_std
15,numeric__clicks_last_30d,0.079597,0.003787
13,numeric__days_with_impressions,0.051590,0.004576
19,numeric__content_age_days,0.028363,0.002605
9,numeric__users_90d,0.023487,0.001773
14,numeric__days_with_sessions,0.022650,0.001834
23,numeric__avg_position,0.018758,0.001760
48,categorical__freshness_tier_0-30,0.012740,0.001276
68,categorical__position_tier_top_3,0.008239,0.001162
3,numeric__word_count,0.006935,0.001076
20,numeric__age_tier_order,0.002869,0.000837


In [79]:
# ### Feature interpretation

# Permutation importance suggests that `clicks_last_30d` was the strongest measured signal in the Logistic Regression model, followed by `days_with_impressions` and `content_age_days`. `users_90d` and `days_with_sessions` were also among the more useful signals.

# These signals are directionally plausible because recent engagement, observed visibility, and content age can be related to whether content is classified as declining. However, the model is associative rather than causal, so these features should not be interpreted as causes of decline.

# The coefficient view also showed mixed directions among related traffic variables. For example, `clicks_last_30d` had a negative coefficient while `sessions_90d` had a positive coefficient. Because these traffic variables can be correlated, individual coefficients should be interpreted cautiously. Permutation importance provides a more useful view of which features contributed to predictive performance.

# No single feature appears suspiciously perfect: the strongest permutation importance is 0.0796 rather than a near-total dependence on one feature. The label-derived fields `trend_direction` and `trend_pct` were excluded from the candidate features.

In [80]:
model_ranked["predicted"] = y_test_pred

false_positives = model_ranked[
    (model_ranked["predicted"] == 1) &
    (model_ranked["actual"] == 0)
].sort_values(
    "predicted_probability",
    ascending=False
)

print("False positives:", len(false_positives))

false_positives.head(10)

False positives: 1531


,content_id,actual,predicted_probability,predicted
21088,content_d896af65d5b6,0,0.958494,1
4495,content_f4c93868660b,0,0.915432,1
4905,content_f0d98be4b42c,0,0.911282,1
20736,content_41baf0722ad9,0,0.908846,1
20401,content_62c74ae619f8,0,0.908155,1
18924,content_a053262db2c0,0,0.901382,1
11887,content_ce59581533ca,0,0.899731,1
11097,content_be9b91f81ef5,0,0.898070,1
10750,content_8725d0090607,0,0.895394,1
4050,content_500bd3907331,0,0.893400,1


In [81]:
error_ids = [
    "content_d896af65d5b6",
    "content_f4c93868660b",
    "content_f0d98be4b42c"
]

error_cases = X_test[
    X_test["content_id"].isin(error_ids)
].copy()

error_cases[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "users_90d",
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d",
        "content_age_days",
        "days_since_last_update",
        "avg_position",
        "engagement_rate",
        "scroll_rate"
    ]
]

,content_id,impressions_90d,clicks_90d,sessions_90d,users_90d,impressions_last_30d,clicks_last_30d,sessions_last_30d,content_age_days,days_since_last_update,avg_position,engagement_rate,scroll_rate
4495,content_f4c93868660b,87433,776,709,683,23246,181,174,421,104,3.4,7.76,14.88
4905,content_f0d98be4b42c,5818,9,9,9,1793,1,1,230,104,5.1,22.22,30.00
21088,content_d896af65d5b6,64718,192,196,188,27874,40,46,445,25,6.0,2.55,29.05


In [82]:
df[
    df["content_id"].isin(error_ids)
][
    ["content_id", "trend_direction", "trend_pct"]
]

,content_id,trend_direction,trend_pct
4495,content_f4c93868660b,stable,-18.8
4905,content_f0d98be4b42c,stable,6.2
21088,content_d896af65d5b6,stable,15.8


In [83]:
# ### Error analysis

# The model produced 1,531 false positives on the held-out test set. The three inspected examples show why some high-confidence predictions can still be wrong.

# - `content_f4c93868660b` received a 0.915 predicted probability of decline, but its observed trend was `stable` at -18.8%. Its recent traffic was substantial, but the trend was just above the threshold used to define `down`, making it a borderline case.
# - `content_f0d98be4b42c` received a 0.911 predicted probability of decline but was `stable` at +6.2%. It had relatively low traffic and engagement, which may have made it resemble declining content to the model.
# - `content_d896af65d5b6` received a 0.958 predicted probability of decline but was `stable` at +15.8%. It had high historical visibility and substantial recent impressions, showing that the model can confuse high-volume patterns with decline.

# These examples show that the model is not a perfect decision rule. It can be especially uncertain around borderline trends or when traffic and engagement signals have patterns associated with declining content. The inspected errors are consistent with the model learning associations among observed search and engagement signals rather than directly observing the target trend.

In [84]:
print("Baseline Precision@20:", baseline_precision_at_20)
print("Model Precision@20:", model_precision_at_20)
print("Test base rate:", base_rate)

Baseline Precision@20: 0.35
Model Precision@20: 0.8
Test base rate: 0.5109524582184002


In [85]:
forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Leakage check:")

for feature in forbidden_features:
    print(
        feature,
        "->",
        feature in feature_columns
    )

Leakage check:
trend_direction -> False
trend_pct -> False
is_declining_label -> False


In [86]:
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", X_train["client_id"].nunique())
print("Test clients:", X_test["client_id"].nunique())

print(
    "Client overlap:",
    len(
        set(X_train["client_id"]) &
        set(X_test["client_id"])
    )
)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.